# Base Files and Source Data Generation

In [ ]:
# package imports

import compgenlnc as cgl

In [ ]:
# setting library
# change settings according to your environment

# base_dir = cgl.config.PACKAGE_DIR
base_dir = "."
cgl.set_base_path(base_dir)

conda_env = cgl.config.conda_environment
cgl.set_conda_env(conda_env)

doc2vec_seed = cgl.config.doc2vec_seed
cgl.set_doc2vec_seed(doc2vec_seed)

log_file = cgl.config.log_file
cgl.set_log_file(log_file)

In [ ]:
# load base files in data directory

# set to load from local or ignore to fetch from repository
basefiles_dir = None

cgl.fetch_lncrna_files()
cgl.fetch_mirna_files()
cgl.fetch_interaction_files()

In [ ]:
# generate the rest of base files

# try different max sizes
maxsize = 500

lncrnas = cgl.load_fasta(cgl.paths.LNCRNA_FASTA, mode=cgl.SEQ_MODE)
matures = cgl.load_fasta(cgl.paths.MAT_MIRNA_FASTA, mode=cgl.SEQ_MODE)
precursors = cgl.load_fasta(cgl.paths.PRE_MIRNA_FASTA, mode=cgl.SEQ_MODE)
lnc_keep = [rec.id for rec in lncrnas if len(rec.seq) <= maxsize]
mir_keep = [rec.id for rec in matures if len(rec.seq) <= maxsize]

# define interaction and index files
cgl.extract_interactions(
    cgl.paths.PAIRS_FILE,
    cgl.paths.PAIRS_FILE_POSITIVE,
    separator=",",
    lnc_list=lnc_keep,
    mir_list=mir_keep,
)
precursors_dict = cgl.load_precursors(cgl.paths.DATA_MIRNA)
interactions = cgl.load_interactions(cgl.paths.PAIRS_FILE, lnc_keep=lnc_keep, mir_keep=mir_keep)
cgl.gen_indexes(interactions, precursors_dict)

lnc_keep = set(cgl.load_indexes()[cgl.LNCRNA_FIELD])
mir_keep = set(cgl.load_indexes()[cgl.PRECURSOR_FIELD])

# define RNA files
cgl.gen_fasta_2d(cgl.paths.LNCRNA_FASTA, cgl.paths.LNCRNA_SS_FASTA, keep=lnc_keep)
cgl.fasta_to_seq_ss(cgl.paths.LNCRNA_FASTA, cgl.paths.LNCRNA_SS_FASTA, cgl.paths.LNCRNA_SEQ_SS_DIR, keep=lnc_keep)

cgl.gen_fasta_2d(cgl.paths.PRE_MIRNA_FASTA, cgl.paths.MIRNA_SS_FASTA, keep=mir_keep)
cgl.fasta_to_seq_ss(cgl.paths.PRE_MIRNA_FASTA, cgl.paths.MIRNA_SS_FASTA, cgl.paths.MIRNA_SEQ_SS_DIR, keep=mir_keep)

lncrnas = set(cgl.index_lncrna())
matures = set(cgl.index_mature_mirna())
precursors = set(cgl.index_precursor_mirna())

# calculate binding zones
cgl.extract_2d_structure_loops_from_list(cgl.paths.LNCRNA_LOOPS_DIR, lncrnas)
cgl.extract_2d_structure_loops_from_list(cgl.paths.MIRNA_LOOPS_DIR, precursors)

# calculate binding zones
cgl.predict_miranda_from_list(cgl.paths.MIRANDA_DIR, lncrnas, matures)
cgl.extract_binding_zone_from_dir(cgl.paths.MIRANDA_DIR, cgl.paths.BINDING_DIR)